# ASR Baseline experiment using Whisper and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for ASR on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using the Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing Word Error Rate, WER)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

import jiwer

model = whisper.load_model("base")

Load Europarl-ST speech dataset

In [3]:
audios = []

with open(CONFIG.europarl_test1_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        audios.append(str(linea).strip())

len(audios)

86

<p style="page-break-after:always;"></p>

Transcribe all the audio data using the Whisper (base) model. The ASR output is stored in hypotheses. At the same time, transcription references are stored into references and translation references into translations.

In [7]:
from maikol_utils.print_utils import print_error
hypotheses = []
references = []
translations = []

for audio in tqdm(audios):
    try:
        with open(os.path.join(CONFIG.europarl_test1_folder, audio, "transcription.tok"), "r", encoding="utf-8") as referencia:
            references.append(referencia.read())
        with open(os.path.join(CONFIG.europarl_test1_folder, audio, "translation_en"), "r", encoding="utf-8") as translation:
            translations.append(translation.read())

        hypotheses.append((model.transcribe(os.path.join(CONFIG.europarl_test1_folder, audio, "audio_clip_diarization.m4a"), language=CONFIG.src_name))['text'])
    except Exception as e:
        print_error(f"Error processing {audio}: {e}")
        hypotheses.append("")
        references.append("")
        translations.append("")

  0%|          | 0/86 [00:00<?, ?it/s]

<p style="page-break-after:always;"></p>

Transcription hypotheses, references and translations are stored into a Pandas dataframe. Show the two first hypotheses.

In [8]:
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references, translation=translations))
pd.set_option('display.max_colwidth', None)
data.head(2)

,hypothesis,reference,translation
0,"A estratégia de Madrid, as aficionadas e, incluso, a Polícia Española, estão sendo maltratados por aceleração europeia de FUOL, mas, em caso de outra senda, particular. Pois, esses órganos federativos agravam as anciones a quem recorrem a justicia ordinaria. Esta concepção medieval, esta lei de lembudo, é incompatible com o direito, com as instituciones europeas, desde queemos de reaccionar e lo caberemos haciendo, pues, esses señores medievales, arbitrarios de orca e cuchillo, ande ponersem a linea com o respeito ao direito e as garantias processares ordinárias de nossa Europa. Nada mais, muchas vezes.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe."
1,"E, graças a Senhor Presidente, Senhor Camisario, o terrorismo subfenome no global e a actuação de la delicência grave organizada também. E, por tanto, todos os medios têm que ser proporcionales e têm que luchar-se com eficacia. Eu me dou uma boa nota de respuestas que eu tenho a perguntas que eram oportunas. É verdade que, se girar entias, é verdade que é um tema delicado, mas não é bem ou certo que é absolutamente inexcusable montar, formar uma resposta globalizada e armonizada. Para alguns, que o terrorismo espilla um pouco lejos, ele espere a preocupa mais nas garantias individuales. Ele me preocupa nas individuales e nas colectivas. E é absolutamente necessário que pensemos por onde podamos. Se vamos empezar por transporte aéreo, onde já as companhias e as cienes dos datos, pensemos por aí. E, sejamos garantias, veamos qual é o ámbito de aplicação, empecemos por relaciones de outras portas internacionales. E, ojo, tendremos que seguir por as interiores, porque nos terroristas, muitas vezes, não virem de fora, se não que virem de dentro. Que se lo preguem em Estados Unidos e se lembram nos demais, que assim é, e assim, tendremos que plantear.","Señor Presidente , señor Comisario , el terrorismo es un fenómeno global y la actuación de la delincuencia grave organizada también , y por tanto los medios tienen que ser proporcionales y hay que luchar con eficacia .\nHe tomado buena nota de las respuestas que ha dado a las preguntas , y eran oportunas : es verdad que hay que exigir garantías , es verdad que es un tema delicado , pero no es menos cierto que es absolutamente inexcusable montar , formar una respuesta globalizada y armonizada .\nA algunos , a los que el terrorismo les queda un poco lejos , les preocupan más las garantías individuales . A mí me preocupan las individuales y las colectivas , y es absolutamente necesario que empecemos por donde podamos . Si hemos de empezar por el transporte aéreo , donde ya las compañías tienen esos datos , empecemos por ahí .\nExijamos garantías , veamos cuál es el ámbito de aplicación , empecemos por las relaciones de los transportes internacionales , y ¡ ojo ! tendremos que seguir por las inte

Hypotheses and references are normalized using the Whisper Basic text standardisation/normalization module

In [9]:
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data.head(2)

,hypothesis,reference,translation,hypothesis_clean,reference_clean,translation_clean
0,"A estratégia de Madrid, as aficionadas e, incluso, a Polícia Española, estão sendo maltratados por aceleração europeia de FUOL, mas, em caso de outra senda, particular. Pois, esses órganos federativos agravam as anciones a quem recorrem a justicia ordinaria. Esta concepção medieval, esta lei de lembudo, é incompatible com o direito, com as instituciones europeas, desde queemos de reaccionar e lo caberemos haciendo, pues, esses señores medievales, arbitrarios de orca e cuchillo, ande ponersem a linea com o respeito ao direito e as garantias processares ordinárias de nossa Europa. Nada mais, muchas vezes.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe.",a estratégia de madrid as aficionadas e incluso a polícia española estão sendo maltratados por aceleração europeia de fuol mas em caso de outra senda particular pois esses órganos federativos agravam as anciones a quem recorrem a justicia ordinaria esta concepção medieval esta lei de lembudo é incompatible com o direito com as instituciones europeas desde queemos de reaccionar e lo caberemos haciendo pues esses señores medievales arbitrarios de orca e cuchillo ande ponersem a linea com o respeito ao direito e as garantias processares ordinárias de nossa europa nada mais muchas vezes,el atlético de madrid los aficionados e incluso la policía española están siendo maltratados por la federación europea de fútbol pero el caso transciende lo particular pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria esta concepción medieval esta ley del embudo es incompatible con el derecho y con las instituciones europeas desde las que hemos de reaccionar lo acabaremos haciendo pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al derecho y las garantías procesales ordinarias de nuestra europa,atlético madrid its fans and even the spanish police are being mistreated by the union of european football associations however the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts this mediaeval concept of one law for me and another for you is contrary to our law and the european institutions we must therefore react in fact we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our europe
1,"E, graças a Senhor Presidente, Senhor Camisario, o terrorismo subfenome no global e a actuação de la delicência grave organizada também. E, por tanto, todos os medios têm que ser proporcionales e têm que luchar-se com eficacia. Eu me dou uma boa nota de respuestas que eu tenho a perguntas que eram oportunas. É verdade que, se girar entias, é verdade que é um tema delicado, mas não é bem o

Finally, we compute the transcription WER using [JIWER](https://openai.com/index/whisper/) which is a simple and fast python package to evaluate ASR performance.

In [10]:

wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER: {wer * 100:.2f} %")

WER: 57.80 %


All the data is stored into a file using 'csv' format

In [12]:
data.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.1_ASR_Whisper_Baseline_Europarl-ST.csv'), encoding='utf-8')